"""
Titanic Survival Prediction - Complete ML Project
Goal: Predict whether a passenger survived using Logistic Regression,
with a Random Forest comparison as a stretch goal.
"""

In [41]:
# ============================================================
# 1. IMPORTS
# ============================================================
import pandas as pd
import numpy as np
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

In [42]:
# ============================================================
# 2. LOAD DATA
# ============================================================
df = sns.load_dataset('titanic')
print("Dataset shape:", df.shape)
print(df.head())
print("\nMissing values:\n", df.isnull().sum())

Dataset shape: (891, 15)
   survived  pclass     sex   age  sibsp  parch     fare embarked  class  \
0         0       3    male  22.0      1      0   7.2500        S  Third   
1         1       1  female  38.0      1      0  71.2833        C  First   
2         1       3  female  26.0      0      0   7.9250        S  Third   
3         1       1  female  35.0      1      0  53.1000        S  First   
4         0       3    male  35.0      0      0   8.0500        S  Third   

     who  adult_male deck  embark_town alive  alone  
0    man        True  NaN  Southampton    no  False  
1  woman       False    C    Cherbourg   yes  False  
2  woman       False  NaN  Southampton   yes   True  
3  woman       False    C  Southampton   yes  False  
4    man        True  NaN  Southampton    no   True  

Missing values:
 survived         0
pclass           0
sex              0
age            177
sibsp            0
parch            0
fare             0
embarked         2
class            0
who  

In [43]:
# ============================================================
# 3. FEATURE / TARGET SELECTION
# ============================================================
y = df['survived']
X = df[['pclass', 'sex', 'age', 'fare', 'sibsp', 'parch']].copy()

# Handle missing values BEFORE encoding/splitting
X['age'] = X['age'].fillna(X['age'].median())
X['fare'] = X['fare'].fillna(X['fare'].median())

# Encode 'sex' as numeric
X = pd.get_dummies(X, columns=['sex'], drop_first=True)

print("\nFinal features used:", list(X.columns))
print("Any remaining NaNs?\n", X.isnull().sum())


Final features used: ['pclass', 'age', 'fare', 'sibsp', 'parch', 'sex_male']
Any remaining NaNs?
 pclass      0
age         0
fare        0
sibsp       0
parch       0
sex_male    0
dtype: int64


In [44]:
# ============================================================
# 4. TRAIN / TEST SPLIT
# ============================================================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [45]:
# ============================================================
# 5. BASELINE (majority class guess)
# ============================================================
baseline_accuracy = max(y_test.mean(), 1 - y_test.mean())
print(f"\nBaseline accuracy (always predict majority class): {baseline_accuracy:.4f}")


Baseline accuracy (always predict majority class): 0.5866


In [46]:
# ============================================================
# 6. LOGISTIC REGRESSION
# ============================================================
log_model = LogisticRegression(max_iter=1000)
log_model.fit(X_train, y_train)
log_pred = log_model.predict(X_test)

log_accuracy = accuracy_score(y_test, log_pred)
print("\n--- Logistic Regression ---")
print("Accuracy:", log_accuracy)
print("Confusion Matrix:\n", confusion_matrix(y_test, log_pred))
print("Classification Report:\n", classification_report(y_test, log_pred))


--- Logistic Regression ---
Accuracy: 0.8100558659217877
Confusion Matrix:
 [[92 13]
 [21 53]]
Classification Report:
               precision    recall  f1-score   support

           0       0.81      0.88      0.84       105
           1       0.80      0.72      0.76        74

    accuracy                           0.81       179
   macro avg       0.81      0.80      0.80       179
weighted avg       0.81      0.81      0.81       179



In [47]:
# ============================================================
# 7. INTERPRETATION - Logistic Regression coefficients
# ============================================================
coef_table = pd.DataFrame({
    'feature': X.columns,
    'coefficient': log_model.coef_[0]
}).sort_values('coefficient', ascending=False)

print("\nLogistic Regression Coefficients (higher = pushes toward 'survived'):")
print(coef_table)


Logistic Regression Coefficients (higher = pushes toward 'survived'):
    feature  coefficient
2      fare     0.003434
1       age    -0.031452
4     parch    -0.125203
3     sibsp    -0.315802
0    pclass    -0.928188
5  sex_male    -2.610598


In [48]:
# ============================================================
# 8. STRETCH GOAL - Random Forest comparison
# ============================================================
rf_model = RandomForestClassifier(random_state=42)
rf_model.fit(X_train, y_train)
rf_pred = rf_model.predict(X_test)

rf_accuracy = accuracy_score(y_test, rf_pred)
print("\n--- Random Forest ---")
print("Accuracy:", rf_accuracy)
print("Confusion Matrix:\n", confusion_matrix(y_test, rf_pred))
print("Classification Report:\n", classification_report(y_test, rf_pred))

importances = pd.DataFrame({
    'feature': X.columns,
    'importance': rf_model.feature_importances_
}).sort_values('importance', ascending=False)

print("\nRandom Forest Feature Importances:")
print(importances)


--- Random Forest ---
Accuracy: 0.8044692737430168
Confusion Matrix:
 [[89 16]
 [19 55]]
Classification Report:
               precision    recall  f1-score   support

           0       0.82      0.85      0.84       105
           1       0.77      0.74      0.76        74

    accuracy                           0.80       179
   macro avg       0.80      0.80      0.80       179
weighted avg       0.80      0.80      0.80       179


Random Forest Feature Importances:
    feature  importance
2      fare    0.291130
5  sex_male    0.280818
1       age    0.257658
0    pclass    0.086700
3     sibsp    0.047756
4     parch    0.035939


In [ ]:


























# ============================================================
# 9. FINAL COMPARISON
# ============================================================
print("\n=== SUMMARY ===")
print(f"Baseline accuracy:            {baseline_accuracy:.4f}")
print(f"Logistic Regression accuracy: {log_accuracy:.4f}")
print(f"Random Forest accuracy:       {rf_accuracy:.4f}")

if log_accuracy > baseline_accuracy:
    print("Logistic Regression beats the baseline — it's learning something real.")
else:
    print("Logistic Regression is close to baseline — may need better features.")

if rf_accuracy > log_accuracy:
    print("Random Forest outperforms Logistic Regression on this split.")
else:
    print("Random Forest does NOT outperform Logistic Regression here — "
          "added complexity isn't paying off on this dataset.")

In [23]:
# Data handling
import pandas as pd
import numpy as np


# Splitting data
from sklearn.model_selection import train_test_split

# The model
from sklearn.linear_model import LogisticRegression

# Evaluation
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# Optional, for the stretch goal (Random Forest comparison)
from sklearn.ensemble import RandomForestClassifier

In [24]:
# load the dataset
import seaborn as sns

df = sns.load_dataset('titanic')
df.head()

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True


In [25]:
# Target variable
y = df['survived']

# Feature variables
X = df[['pclass', 'sex', 'age', 'fare', 'sibsp', 'parch']]

In [26]:
X['sex'] = X['sex'].map({'male': 0, 'female': 1})

In [27]:
X = df[['pclass', 'sex', 'age', 'fare', 'sibsp', 'parch']].copy()
X['sex'] = X['sex'].map({'male': 0, 'female': 1})

In [28]:
X.isnull().sum()

pclass      0
sex         0
age       177
fare        0
sibsp       0
parch       0
dtype: int64

In [29]:
X.info()

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 6 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   pclass  891 non-null    int64  
 1   sex     891 non-null    int64  
 2   age     714 non-null    float64
 3   fare    891 non-null    float64
 4   sibsp   891 non-null    int64  
 5   parch   891 non-null    int64  
dtypes: float64(2), int64(4)
memory usage: 41.9 KB


0      22.000000
1      38.000000
2      26.000000
3      35.000000
4      35.000000
         ...    
886    27.000000
887    19.000000
888    29.699118
889    26.000000
890    32.000000
Name: age, Length: 891, dtype: float64

In [30]:
X['age'] = X.groupby('pclass')['age'].transform(lambda x: x.fillna(x.median()))

In [31]:
X.isnull().sum()

pclass    0
sex       0
age       0
fare      0
sibsp     0
parch     0
dtype: int64

In [32]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [33]:
X_train.shape, X_test.shape

((712, 6), (179, 6))

In [34]:
model = LogisticRegression()
model.fit(X_train, y_train)


,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary <random_state>` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default sol

In [35]:
y_pred = model.predict(X_test)

In [36]:
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

Accuracy: 0.8268156424581006

Confusion Matrix:
 [[93 12]
 [19 55]]

Classification Report:
               precision    recall  f1-score   support

           0       0.83      0.89      0.86       105
           1       0.82      0.74      0.78        74

    accuracy                           0.83       179
   macro avg       0.83      0.81      0.82       179
weighted avg       0.83      0.83      0.83       179



In [37]:
y_test.value_counts(normalize=True)

survived
0    0.586592
1    0.413408
Name: proportion, dtype: float64

In [38]:
coefficients = pd.DataFrame({
    'feature': X.columns,
    'coefficient': model.coef_[0]
})
print(coefficients)

  feature  coefficient
0  pclass    -0.993154
1     sex     2.604240
2     age    -0.034016
3    fare     0.003177
4   sibsp    -0.324744
5   parch    -0.121756


In [39]:
rf_model = RandomForestClassifier(random_state=42)
rf_model.fit(X_train, y_train)

rf_pred = rf_model.predict(X_test)

print("Random Forest Accuracy:", accuracy_score(y_test, rf_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, rf_pred))

Random Forest Accuracy: 0.8100558659217877

Confusion Matrix:
 [[91 14]
 [20 54]]


In [40]:
importances = pd.DataFrame({
    'feature': X.columns,
    'importance': rf_model.feature_importances_
}).sort_values('importance', ascending=False)
print(importances)

  feature  importance
3    fare    0.303974
1     sex    0.272034
2     age    0.253599
0  pclass    0.082014
4   sibsp    0.052022
5   parch    0.036357
